In [1]:
import os
import sys
import pandas as pd
import plotly.express as px
import numpy as np
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from collections import Counter
import matplotlib.pyplot as plt

In [2]:
parent_directory=os.path.dirname(os.getcwd())
path=os.path.join(parent_directory,"src")
sys.path.append(parent_directory)
import src.cleaning_data as cln
BASE_DIR = os.path.dirname(os.getcwd())
proccessed_file=os.path.join(BASE_DIR,"data","processed","processed.csv")

raw_df = pd.read_csv(proccessed_file).copy()

processed_ventas_df = cln.cleaning_data_frame_by_category(raw_df,"venta",min_outliner=1000,max_outliner=500000)
processed_alquiler_df = cln.cleaning_data_frame_by_category(raw_df,"alquiler",min_outliner=10,max_outliner=1000)
processed_permutas_df = cln.cleaning_data_frame_by_category(raw_df,"permuta",min_outliner=1000,max_outliner=100000)

processed_ventas_df["Fecha"] = pd.to_datetime(processed_ventas_df["Fecha"], errors="coerce")

initial_count = len(processed_ventas_df)
processed_ventas_df = processed_ventas_df.dropna(subset=["Fecha"])

processed_ventas_df["Año"] = processed_ventas_df["Fecha"].dt.year
processed_ventas_df["Mes"] = processed_ventas_df["Fecha"].dt.month
processed_ventas_df["Año-Mes"] = processed_ventas_df["Fecha"].dt.strftime("%Y-%m")

precio_mensual = processed_ventas_df.groupby("Año-Mes", as_index=False).agg(
    Precio_Promedio=("Precio", "mean"),
    Cantidad_Propiedades=("Precio", "count")
)
precio_anual = processed_ventas_df.groupby("Año", as_index=False).agg(
    Precio_Promedio=("Precio", "mean"),
    Cantidad_Propiedades=("Precio", "count")
)
processed_ventas_df = cln.clean_locations(raw_df)

In [3]:
df_ventas_Casas = processed_ventas_df[processed_ventas_df["Tipo"] == "casa"].copy()
df_ventas_Apartamentos = processed_ventas_df[processed_ventas_df["Tipo"] == "apartamento"].copy()
df_permutas_Casas = processed_permutas_df[processed_permutas_df["Tipo"] == "casa"].copy()
df_permutas_Apartamentos = processed_permutas_df[processed_permutas_df["Tipo"] == "apartamento"].copy()
df_alquiler_Casas = processed_alquiler_df[processed_alquiler_df["Tipo"] == "casa"].copy()
df_alquiler_Apartamentos = processed_alquiler_df[processed_alquiler_df["Tipo"] == "apartamento"].copy()

In [4]:
def analizar_municipio(processed_ventas_df, municipio):

    COLOR_PRINCIPAL = "#F3D056"
    COLOR_FONDO = "#001734"

    df_municipio = processed_ventas_df[processed_ventas_df["Municipio"] == municipio].copy()

    if "Fecha" in df_municipio.columns:
    
        df_municipio["Fecha"] = pd.to_datetime(df_municipio["Fecha"])
        df_municipio["Año-Mes"] = df_municipio["Fecha"].dt.to_period("M").astype(str)
        
        precio_mensual_municipio = df_municipio.groupby("Año-Mes", as_index=False)["Precio"].mean()
        
        fig_line = px.line(
            precio_mensual_municipio,
            x="Año-Mes",
            y="Precio",
            title=f"Evolución de Precios en {municipio}",
            labels={"Precio": "Precio Promedio (USD)", "Año-Mes": "Periodo"},
            markers=True
        )
        
        fig_line.update_traces(
            line=dict(color=COLOR_PRINCIPAL, width=3),
            marker=dict(color=COLOR_PRINCIPAL, size=8)
        )
        
        fig_line.update_layout(
            template="plotly_dark",
            plot_bgcolor=COLOR_FONDO,
            paper_bgcolor=COLOR_FONDO,
            font=dict(color="white"),
            title_font=dict(color=COLOR_PRINCIPAL),
            height=500
        )
        fig_line.show()
    

def analizar_municipios(processed_ventas_df, municipios):
    
    for municipio in municipios:
        analizar_municipio(processed_ventas_df, municipio)

analizar_municipio(processed_ventas_df, "Boyeros")
analizar_municipios(processed_ventas_df, ["Playa", "Habana del Este", "Centro Habana", "Guanabacoa"])

In [5]:
def proporcion_ventas_permutas(municipio=None, fecha_inicio=None, fecha_fin=None):
    
    COLOR_VENTAS = "#F3D056"  
    COLOR_PERMUTAS = "#2A4A6B"  
    COLOR_FONDO = "#001734" 
    
    ventas_count = len(processed_ventas_df)
    permutas_count = len(processed_permutas_df)
    
    data = {
        "Tipo": ["Ventas", "Permutas"],
        "Cantidad": [ventas_count, permutas_count]
    }
    VP_df = pd.DataFrame(data)
    
    total = ventas_count + permutas_count
    VP_df["Porcentaje"] = VP_df["Cantidad"] / total * 100
    
    fig = px.pie(
        VP_df,
        names="Tipo",
        values="Cantidad",
        title=f"Proporción de Ventas vs Permutas{" en " + municipio if municipio else ""}",
        color="Tipo",
        color_discrete_map={
            "Ventas": COLOR_VENTAS,
            "Permutas": COLOR_PERMUTAS
        }
    )

    fig.update_layout(
        template="plotly_dark",
        plot_bgcolor=COLOR_FONDO,
        paper_bgcolor=COLOR_FONDO,
        font=dict(color="white"),
        title_font=dict(size=24, color=COLOR_VENTAS),
        legend_title_text="Tipo de Operación",
        annotations=[dict(
            text=f"Total: {total}",
            x=0.5, y=0.5,
            font_size=20,
            showarrow=False,
            font_color=COLOR_VENTAS
        )]
    )
    
    fig.update_traces(
        textinfo="percent+label",
        textposition="inside",
        textfont=dict(color="white", size=14),
        hovertemplate="<b>%{label}</b><br>%{value} anuncios (%{percent})"
    )
    
    fig.show()
    
proporcion_ventas_permutas(municipio="La Habana Vieja")

In [6]:
Prec_Prom_Anual_fig = px.bar(
    precio_anual,
    x='Año',
    y='Precio_Promedio',
    title='Precio Promedio Anual',
    color='Precio_Promedio'
)
Prec_Prom_Anual_fig.update_layout(
    xaxis_title='Periodo',
    yaxis_title='Precio Promedio (USD)',
    template='plotly_dark',  
    plot_bgcolor='#001734', 
    paper_bgcolor='#001734',  
    font=dict(color='white'),  
    title_font=dict(color='white'),  
)
Prec_Prom_Anual_fig.show()

In [7]:
Prec_Mens_fig = px.line(
    precio_mensual,
    x='Año-Mes',
    y='Precio_Promedio',
    title='Evolución Mensual de Precios de Propiedades en Venta',
    labels={'Precio_Promedio': 'Precio Promedio (USD)', 'Año-Mes': 'Mes'},
    markers=True
)
Prec_Mens_fig.update_layout(
    xaxis_title='Periodo',
    yaxis_title='Precio Promedio (USD)',
    template='plotly_dark',  
    plot_bgcolor='#001734',  
    paper_bgcolor='#001734',  
    font=dict(color='white'),  
    title_font=dict(color='white'), 
)

Prec_Mens_fig.update_traces(
    line=dict(color='#F3D056', width=3),  
    marker=dict(color='#F3D056', size=8), 
    textposition='top center'
)

Prec_Mens_fig.show()